In [ ]:
# Install missing libraries
!pip install pypdf sentence-transformers

import os, zipfile
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, util

# -----------------------------
# Config
# -----------------------------
ZIP_FILE   = "/content/Education.zip"       # resumes archive
JD_PDF     = "/content/20_Job_Descriptions (2).pdf" # job descriptions
THRESHOLD  = 0.60
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# -----------------------------
# Helpers
# -----------------------------
def read_pdf(path):
    """Extract text from a PDF file."""
    try:
        return " ".join(p.extract_text() or "" for p in PdfReader(path).pages)
    except Exception as e:
        print(f" Error reading {path}: {e}")
        return ""

def get_resumes(zip_file):
    """Return list of (filename, text) for resumes inside ZIP."""
    resumes = []
    if not zipfile.is_zipfile(zip_file):
        raise ValueError(f"{zip_file} is not a valid ZIP file")

    with zipfile.ZipFile(zip_file, "r") as z:
        z.extractall("resumes")

    resume_paths = [
        os.path.join(r, f)
        for r, _, fs in os.walk("resumes")
        for f in fs if f.endswith(".pdf")
    ]

    for p in resume_paths:
        text = read_pdf(p)
        if text.strip():
            resumes.append((os.path.basename(p), text))
    return resumes

def get_job_descriptions(jd_pdf):
    """Return list of (title, text) for job descriptions (1 per page)."""
    jd_list = []
    for p in PdfReader(jd_pdf).pages:
        txt = p.extract_text()
        if txt:
            title = txt.strip().split("\n")[0]
            jd_list.append((title, txt))
    return jd_list

# -----------------------------
# Load data
# -----------------------------
resumes = get_resumes(ZIP_FILE)
jd_list = get_job_descriptions(JD_PDF)

print(f"Resumes loaded: {len(resumes)} | Job Descriptions loaded: {len(jd_list)}")

# -----------------------------
# Embeddings & Similarity
# -----------------------------
model = SentenceTransformer(MODEL_NAME)

resume_embeddings = model.encode([text for _, text in resumes], convert_to_tensor=True)
jd_embeddings     = model.encode([text for _, text in jd_list], convert_to_tensor=True)

cosine_scores = util.cos_sim(resume_embeddings, jd_embeddings)

# -----------------------------
# Report matches
# -----------------------------
for i, (res_name, _) in enumerate(resumes):
    print(f"\n Resume: {res_name}")
    for j, (jd_title, _) in enumerate(jd_list):
        score = float(cosine_scores[i][j])
        if score >= THRESHOLD:
            print(f" Matches JD '{jd_title}' with score {score:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 27.0 MB/s eta 0:00:00
✅ Resumes loaded: 5000 | Job Descriptions loaded: 20


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Streaming output truncated to the last 5000 lines.

📄 Resume: Resume_030309_Catherine_Bryan.pdf

📄 Resume: Resume_034840_Jeffrey_Williams.pdf

📄 Resume: Resume_034944_Melissa_Garcia.pdf

📄 Resume: Resume_031667_Joyce_Lopez.pdf

📄 Resume: Resume_033163_Tamara_Valdez.pdf

📄 Resume: Resume_030351_Justin_Love.pdf

📄 Resume: Resume_031930_Brittany_Torres.pdf

📄 Resume: Resume_032279_Daniel_Myers.pdf

📄 Resume: Resume_032303_Maria_Allen.pdf

📄 Resume: Resume_033820_Gregory_Jones.pdf

📄 Resume: Resume_031045_Justin_Reed.pdf

📄 Resume: Resume_032529_Megan_Torres.pdf

📄 Resume: Resume_033982_Amy_Lopez.pdf

📄 Resume: Resume_032653_Thomas_Wall.pdf

📄 Resume: Resume_033031_Lisa_Washington.pdf

📄 Resume: Resume_033630_William_Miles.pdf

📄 Resume: Resume_030398_Rebecca_Jones.pdf

📄 Resume: Resume_031689_William_Palmer.pdf

📄 Resume: Resume_031340_Anthony_Sloan.pdf

📄 Resume: Resume_032634_Regina_Moore.pdf

📄 Resume: Resume_032675_Miranda_Castillo.pdf

📄 Resume: Resume_031070_Paul_Allen.pdf

📄 Resume